In [1]:
import random
from distributed_processing.utils import fsclient
from distributed_processing.async_result import gather
import fs_structs
from dotenv import load_dotenv
from os import getenv

In [2]:
load_dotenv()
NS_PATH = getenv("NS_PATH")

In [3]:
import logging
logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
#logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [4]:
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)

In [5]:
client.update_registry_cache()

In [6]:
ns = fs_structs.structs.FSNamespace(NS_PATH)

In [7]:
ns.registry.keys()

['method_queues_hola',
 'nservers',
 'method_queues_kill_all_processes',
 'method_queues_kill_processes',
 'method_queues_dic',
 'method_queues_sleep',
 'method_queues_mul',
 'method_queues_eval_py_function',
 'workers_queue_requests_cola_2',
 'method_queues_lista',
 'method_queues_add',
 'workers_queue_requests_fs_server_1',
 'method_queues_div',
 'method_queues_tupla',
 'workers_queue_requests_node_1',
 'workers_queue_requests_cola_1',
 'workers_queue_requests_fs_server_3',
 'method_queues_kill_process',
 'workers_queue_requests_py_eval',
 'workers_queue_requests_fs_server_4',
 'method_queues_info',
 'nclients',
 'workers_queue_requests_fs_server_5',
 'method_queues_cleanup',
 'workers_queue_requests_fs_server_6',
 'method_queues_create_worker',
 'workers_queue_requests_fs_server_2',
 'method_queues_list_processes']

In [8]:
z1 = client.rpc_sync("create_worker", ["worker1"])
z2 = client.rpc_sync("create_worker", ["worker1"])
z3 = client.rpc_sync("create_worker", ["worker1"])

(z1, z2, z3)

((49708, 'worker1', 'fs_server_7'),
 (49726, 'worker1', 'fs_server_8'),
 (49744, 'worker1', 'fs_server_9'))

In [9]:
client.rpc_sync("list_processes", [])

[(48040, 'worker1', 'fs_server_1'),
 (48061, 'worker1', 'fs_server_2'),
 (48089, 'worker1', 'fs_server_3'),
 (49583, 'worker1', 'fs_server_4'),
 (49601, 'worker1', 'fs_server_5'),
 (49619, 'worker1', 'fs_server_6'),
 (49708, 'worker1', 'fs_server_7'),
 (49726, 'worker1', 'fs_server_8'),
 (49744, 'worker1', 'fs_server_9')]

In [10]:
client.rpc_sync("kill_process", [z2])

False

In [11]:
client.rpc_sync("list_processes", [])

[(48040, 'worker1', 'fs_server_1'),
 (48061, 'worker1', 'fs_server_2'),
 (48089, 'worker1', 'fs_server_3'),
 (49583, 'worker1', 'fs_server_4'),
 (49601, 'worker1', 'fs_server_5'),
 (49619, 'worker1', 'fs_server_6'),
 (49708, 'worker1', 'fs_server_7'),
 (49726, 'worker1', 'fs_server_8'),
 (49744, 'worker1', 'fs_server_9')]

In [12]:
y = client.rpc_async("add", [1, 0])

In [13]:
y.get()

1

In [14]:
client.check_registry ="always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [15]:
client.check_registry ="never" # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


Error -32601 : The method does not exist/is not available.

 NoneType: None



In [16]:
client.check_registry ="never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

'Esto es un error'

In [17]:
client.check_registry ="cache"

In [18]:
x = client.rpc_async("div", [1, 0])

In [19]:
try:
    x.get()
except Exception as e:
    print(e)

Error -32603 : Internal RPC error.

 Traceback (most recent call last):
  File "/home/augusto/python/py_distributed_processing/distributed_processing/worker.py", line 472, in process_single_request
    result = self._exec_method_in_queue(
        dispatched_to, request["method"], args, kwargs
    )
  File "/home/augusto/python/py_distributed_processing/distributed_processing/worker.py", line 394, in _exec_method_in_queue
    return funcs[method](*args, **kwargs)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/augusto/python/py_distributed_processing/examples/filesystem/fs_worker_multi.py", line 28, in div
    return x / y
           ~~^~~
ZeroDivisionError: division by zero



In [20]:
#x = client.rpc_sync("div", [1, 0])

In [21]:
client.rpc_sync("add", [1, 1])

2

In [22]:
def f(x,y): return x + y

y = client.rpc_async_fn(f, [1, 2.0])

In [23]:
y.get()

3.0

In [24]:
client.rpc_sync_fn(f, [3.0, 2.0])

5.0

In [25]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(),random.random()], {})
    print(t)
    tp.append(t)
    fs.append(client.rpc_async(t[0], t[1]))

t = ("sleep", [10], {})
tp.append(t)
fs.append(client.rpc_async(t[0], t[1]))



('tupla', [0.38164851223588614, 0.2929384833210086], {})
('lista', [0.04042264438406129, 0.34360905293707733], {})
('div', [0.02783792529138307, 0.6902814104181256], {})
('tupla', [0.4403257205889759, 0.9317300527542166], {})
('mul', [0.3328065921015778, 0.44459863982612935], {})
('add', [0.6840136124648343, 0.3308397305388361], {})
('div', [0.17580508828903962, 0.7059298172290523], {})
('dic', [0.31335847125799554, 0.22386395336013776], {})
('tupla', [0.4400563621613832, 0.8905243819440029], {})
('dic', [0.2655723362923268, 0.5138300673986715], {})


In [26]:
resultados = [f.get() for f in fs]
resultados

[(0.38164851223588614, 0.2929384833210086),
 [0.04042264438406129, 0.34360905293707733],
 0.04032837169194625,
 (0.4403257205889759, 0.9317300527542166),
 0.14796535817353093,
 1.0148533430036704,
 0.24904046266117177,
 {'a': 0.31335847125799554, 'b': [0.31335847125799554, 0.22386395336013776]},
 (0.4400563621613832, 0.8905243819440029),
 {'a': 0.2655723362923268, 'b': [0.2655723362923268, 0.5138300673986715]},
 None]

In [27]:
fs = client.rpc_multi_async(tp)

In [28]:
# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory. 
# If there are responses available in the queue or in the cache 
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING']

In [29]:
client.wait_responses(timeout=2)

['fs_client_3:35']

In [30]:
[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'PENDING']

In [31]:
client.wait_responses()

[]

In [32]:
# AsynResult.status updates information

[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK']

In [33]:
try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [34]:
client._update_cache_with_all_available_responses()

In [35]:
client.pending

{}

In [36]:
[f.get() for f in fs]

[(0.38164851223588614, 0.2929384833210086),
 [0.04042264438406129, 0.34360905293707733],
 0.04032837169194625,
 (0.4403257205889759, 0.9317300527542166),
 0.14796535817353093,
 1.0148533430036704,
 0.24904046266117177,
 {'a': 0.31335847125799554, 'b': [0.31335847125799554, 0.22386395336013776]},
 (0.4400563621613832, 0.8905243819440029),
 {'a': 0.2655723362923268, 'b': [0.2655723362923268, 0.5138300673986715]},
 None]

In [37]:
fs = client.rpc_batch_async(tp)

In [38]:
[f.get() for f in fs]

[(0.38164851223588614, 0.2929384833210086),
 [0.04042264438406129, 0.34360905293707733],
 0.04032837169194625,
 (0.4403257205889759, 0.9317300527542166),
 0.14796535817353093,
 1.0148533430036704,
 0.24904046266117177,
 {'a': 0.31335847125799554, 'b': [0.31335847125799554, 0.22386395336013776]},
 (0.4400563621613832, 0.8905243819440029),
 {'a': 0.2655723362923268, 'b': [0.2655723362923268, 0.5138300673986715]},
 None]

In [39]:
client.rpc_batch_sync(tp)

[(0.38164851223588614, 0.2929384833210086),
 [0.04042264438406129, 0.34360905293707733],
 0.04032837169194625,
 (0.4403257205889759, 0.9317300527542166),
 0.14796535817353093,
 1.0148533430036704,
 0.24904046266117177,
 {'a': 0.31335847125799554, 'b': [0.31335847125799554, 0.22386395336013776]},
 (0.4400563621613832, 0.8905243819440029),
 {'a': 0.2655723362923268, 'b': [0.2655723362923268, 0.5138300673986715]},
 None]

In [40]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(),random.random()], {})
    print(t)
    tp.append(t)

('div', [0.6025492130758071, 0.700678408290431], {})
('div', [0.33299504993120166, 0.6322702753923356], {})
('div', [0.5818409486154067, 0.8612991149068299], {})
('add', [0.27568578417752243, 0.41655523847752796], {})
('div', [0.21759975249083652, 0.9191143284115307], {})
('div', [0.6584751591377235, 0.9041488892309187], {})
('add', [0.10031774219397427, 0.004574306174557896], {})
('fake', [0.620364917912896, 0.3556789576265146], {})
('add', [0.31020326018380595, 0.34366611801239266], {})
('mul', [0.4286299024062612, 0.5260332829014144], {})


In [41]:
tp

[('div', [0.6025492130758071, 0.700678408290431], {}),
 ('div', [0.33299504993120166, 0.6322702753923356], {}),
 ('div', [0.5818409486154067, 0.8612991149068299], {}),
 ('add', [0.27568578417752243, 0.41655523847752796], {}),
 ('div', [0.21759975249083652, 0.9191143284115307], {}),
 ('div', [0.6584751591377235, 0.9041488892309187], {}),
 ('add', [0.10031774219397427, 0.004574306174557896], {}),
 ('fake', [0.620364917912896, 0.3556789576265146], {}),
 ('add', [0.31020326018380595, 0.34366611801239266], {}),
 ('mul', [0.4286299024062612, 0.5260332829014144], {})]

In [42]:
client.check_registry ="never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [43]:
[f.safe_get() for f in fs]

[0.8599511643950224,
 0.5266656727213247,
 0.6755387745618974,
 0.6922410226550504,
 0.23674938553825578,
 0.7282817763541481,
 0.10489204836853216,
 None,
 0.6538693781961986,
 0.22547359471247844]

In [44]:
client.responses

{}

In [45]:
client.check_registry ="cache"

In [46]:
try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

No common queue for batch request.


In [47]:
try:    
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

No common queue for batch request.


In [48]:
client.check_registry="never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [49]:
try:
    x.get()
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [50]:
y = client.rpc_async("add", [1, 0])

In [51]:
y.get(5)

1

In [52]:
def f(x,y): return x + y

client.check_registry = "never" # "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn 
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [53]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [54]:
y.get()

3.0

In [55]:
[k for k in client.responses.keys()]

[]

In [56]:
client.clean_used()

In [57]:
[k for k in client.responses.keys()]

[]

In [58]:
resultados

[(0.38164851223588614, 0.2929384833210086),
 [0.04042264438406129, 0.34360905293707733],
 0.04032837169194625,
 (0.4403257205889759, 0.9317300527542166),
 0.14796535817353093,
 1.0148533430036704,
 0.24904046266117177,
 {'a': 0.31335847125799554, 'b': [0.31335847125799554, 0.22386395336013776]},
 (0.4400563621613832, 0.8905243819440029),
 {'a': 0.2655723362923268, 'b': [0.2655723362923268, 0.5138300673986715]},
 None]

In [59]:
client.rpc_sync_fn(print, ["hola"])

In [60]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("sleep", "sleep"))
    t = (fn, [15], {})
    print(t)
    tp.append(t)

('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})


In [61]:
client.check_registry="never"
fs = client.rpc_multi_async(tp, retry=True)

In [62]:
from time import time

def gather_kk(arl, timeout=None, step=5, max_dts=0):
    # Update to reset delta times
    pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())] 
    pending_ids = [ar.id for ar in pending_ars]
    if fs[0]._client.wait_responses(pending_ids, timeout=0) == []:
        return

    N = len(arl)

    max_dts = max_dts

    t_0 = time()

    i = 0
    # ok() and failed() don't update.
    # pending() updates, every time is called, all the ids 
    # that have a response available in the queue 
    pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())]
    pending_ars0= pending_ars
    pending_ids = [ar.id for ar in pending_ars]
    t_prev = time()
    n_pending_prev = len(pending_ars)
    n_pending_0 = n_pending_prev    
    while (time() - t_0) <= timeout:
        try:
            if fs[0]._client.wait_responses(pending_ids, timeout=5) == []:
                return
        except TimeoutError:
            pass
        dts = [ar.finished_time - t_0 for ar in pending_ars0 if (ar.ok() or ar.failed())]
        if len(dts)>0:
            #max_dts = max(max_dts, max(dts))
            max_dts = max(dts)    
        pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())] 
        pending_ids = [ar.id for ar in pending_ars]
        if len(pending_ars) < n_pending_prev:
            t_prev = time()
            n_pending_prev = len(pending_ars)  

        avg =  (time()-t_0)/(n_pending_0 -len(pending_ars)) if (n_pending_0 -len(pending_ars))>0 else 0

        print(f"{i}: seconds {time()-t_0}s, AR recovered {N-len(pending_ars)}, \
                AR left {len(pending_ars)}, max delta {max_dts}, avg {avg}")
        i += 0


In [63]:
gather(fs, 30, 5)

True

In [64]:
[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK']

In [65]:
[(time() if f.finished_time is None else f.finished_time) -f.creation_time for f in fs]

[15.06894588470459,
 15.06599760055542,
 15.064809083938599,
 15.063655853271484,
 15.062505960464478,
 15.166239023208618,
 15.166918516159058,
 15.167349100112915,
 15.167710065841675,
 30.038547039031982]

In [66]:
[f.finished_time for f in fs]

[1784153440.8517425,
 1784153440.8523707,
 1784153440.852829,
 1784153440.853274,
 1784153440.853701,
 1784153440.9599597,
 1784153440.9623501,
 1784153440.9642646,
 1784153440.966126,
 1784153455.8384297]

In [67]:
client.pending

{}

In [68]:
fs[3].status

'OK'

In [69]:
fs[4].retry()

False

In [70]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

[False, False, False, False, False, False, False, False, False, False]

In [71]:
client.check_registry="always"
client.all_queues_for_method("info")

['fs_server_6',
 'fs_server_4',
 'fs_server_3',
 'fs_server_8',
 'fs_server_7',
 'fs_server_2',
 'fs_server_1',
 'fs_server_5',
 'fs_server_9']

In [72]:
client.connector.all_queues_for_method("info")

['requests_fs_server_6',
 'requests_fs_server_4',
 'requests_fs_server_3',
 'requests_fs_server_8',
 'requests_fs_server_7',
 'requests_fs_server_2',
 'requests_fs_server_1',
 'requests_fs_server_5',
 'requests_fs_server_9']

In [73]:
client.update_registry_cache()

In [74]:
client.check_registry="Never"
client.all_queues_for_method("hola")

['cola_1']

In [75]:
client.check_registry="always"
a = client.rpc_async("info")

In [76]:
client.rpc_sync("div", [2,1], timeout=10)

2.0

In [77]:
x = a.get()

In [78]:
client.registry

<bound method Client.registry of <distributed_processing.client.Client object at 0x7f5441d6cd70>>

In [79]:
def registry_one_step(x):
    worker_id = x[0]
    registry = {}
    for queue, methods in x[1].items():
        for method in methods:
            if method in registry:
                registry[method].append(queue)
            else:
                registry[method] = [queue]
    return registry 

def queues_workers_one_step(x):
    worker_id = x[0]
    registry = {}
    for queue in x[1]:
        if queue in registry:
            registry[queue].append(worker_id)
        else:
            registry[queue] = [worker_id]
    return registry 

def all_workers_for_queue():
    # all requests queues for "info" method
    # One queue per worker

    rr = {}
    qq = {}
    all_worker_queues = client.connector.all_queues_for_method("info")
    for worker in all_worker_queues:
        try:
            x = client.rpc_sync("info", timeout=10)
        except TimeoutError:
            x = None
        
        if x is not None:
            for method, qs in registry_one_step(x).items():
                if method not in rr:
                    rr[method] = qs
                else:
                    rr[method] += qs
            for q, ws in queues_workers_one_step(x).items():
                if q not in qq:
                    qq[q] = ws
                else:
                    qq[q] += ws
    return rr, qq
            

In [80]:
registry_one_step(x)

{'info': ['requests_fs_server_4'],
 'div': ['requests_cola_1'],
 'tupla': ['requests_cola_1'],
 'sleep': ['requests_cola_1'],
 'dic': ['requests_cola_1'],
 'lista': ['requests_cola_1'],
 'mul': ['requests_cola_1'],
 'add': ['requests_cola_1'],
 'hola': ['requests_cola_2'],
 'eval_py_function': ['requests_py_eval']}

In [81]:
queues_workers_one_step(x)

{'requests_fs_server_4': ['fs_server_4'],
 'requests_cola_1': ['fs_server_4'],
 'requests_cola_2': ['fs_server_4'],
 'requests_py_eval': ['fs_server_4']}

In [82]:
r, q = all_workers_for_queue()

In [83]:
r

{'info': ['requests_fs_server_8',
  'requests_fs_server_9',
  'requests_fs_server_8',
  'requests_fs_server_1',
  'requests_fs_server_9',
  'requests_fs_server_4',
  'requests_fs_server_9',
  'requests_fs_server_4',
  'requests_fs_server_9'],
 'div': ['requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1'],
 'tupla': ['requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1'],
 'sleep': ['requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1'],
 'dic': ['requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'requests_cola_1',
  'reque

In [84]:
q

{'requests_fs_server_8': ['fs_server_8', 'fs_server_8'],
 'requests_cola_1': ['fs_server_8',
  'fs_server_9',
  'fs_server_8',
  'fs_server_1',
  'fs_server_9',
  'fs_server_4',
  'fs_server_9',
  'fs_server_4',
  'fs_server_9'],
 'requests_cola_2': ['fs_server_8',
  'fs_server_9',
  'fs_server_8',
  'fs_server_1',
  'fs_server_9',
  'fs_server_4',
  'fs_server_9',
  'fs_server_4',
  'fs_server_9'],
 'requests_py_eval': ['fs_server_8',
  'fs_server_9',
  'fs_server_8',
  'fs_server_1',
  'fs_server_9',
  'fs_server_4',
  'fs_server_9',
  'fs_server_4',
  'fs_server_9'],
 'requests_fs_server_9': ['fs_server_9',
  'fs_server_9',
  'fs_server_9',
  'fs_server_9'],
 'requests_fs_server_1': ['fs_server_1'],
 'requests_fs_server_4': ['fs_server_4', 'fs_server_4']}

In [85]:
t = (1,2,3,4)

In [86]:
t[:6]

(1, 2, 3, 4)

In [87]:
t[:5]

(1, 2, 3, 4)